In [ ]:
import os
import shutil
import zipfile
import pandas as pd

# === CONFIGURATION ===
zip_file = 'your_zip_file.zip'  # Update with your actual ZIP file name
extract_dir = 'di_PIPE'
output_dir = os.path.join(extract_dir, 'Filtered_BOQs_DI')
keywords = ['di pipe', 'ductile iron pipe']

# === STEP 1: EXTRACT ZIP FILE ===
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print(f"📦 Extracted ZIP to: {extract_dir}")

# === STEP 2: CREATE OUTPUT DIRECTORY ===
os.makedirs(output_dir, exist_ok=True)

# === STEP 3: SCAN EXCEL FILES FOR KEYWORDS ===
for root, _, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith(('.xls', '.xlsx')):
            file_path = os.path.join(root, file)
            try:
                # Use correct engine
                engine = 'openpyxl' if file.endswith('.xlsx') else 'xlrd'
                df = pd.read_excel(file_path, engine=engine)
                df.columns = df.columns.map(str)

                # Check text columns for keywords
                for col in df.select_dtypes(include='object').columns:
                    if df[col].astype(str).str.lower().str.contains('|'.join(keywords), na=False).any():
                        shutil.copy(file_path, output_dir)
                        print(f"✅ DI pipe found in: {file_path}")
                        break  # No need to check more columns
            except Exception as e:
                print(f"❌ Skipped {file_path} due to error: {e}")

# === COMPLETION ===
print(f"\n🏁 Done! Filtered BOQ files are saved in: {output_dir}")
